# Session 9 — Your turn: Transfer learning: reusing a trained network

Two things that decide whether a transfer-learning result can be believed, with no network to train: how the photos are split, and how much a test accuracy on 300 photos can move.

Run the cells from the top. Replace every `...` with your own code, then run the check below it. A check prints PASS or FAIL with a hint; you can run a task as often as you like.

In [ ]:
import numpy as np, pandas as pd
from sklearn.model_selection import GroupShuffleSplit, train_test_split

rng = np.random.default_rng(0)
photos = pd.DataFrame({"source_id": np.repeat(np.arange(120), 4)})                              # 120 original photos, each saved with 3 augmented copies
photos["file"] = [f"flower_{s:03d}_{c}.jpg" for s, c in zip(photos.source_id, np.tile(np.arange(4), 120))]
photos["flower"] = rng.choice(["daisy", "rose", "tulip"], 120)[photos.source_id]
print(len(photos), "files from", photos.source_id.nunique(), "original photos")
from aij_check import check, close, summary

## Task 1: Split by photo, not by file

Four files of one table row group are copies of the same photo. First split the **files** at random into 75% train and 25% test with `train_test_split(photos, test_size=0.25, random_state=0)` and count the test files whose original photo also has a file in the training set: `leaked_files`.
Then do the split properly with `GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=0)` using `source_id` as the group, giving `train_rows` and `test_rows` (dataframes), and `leaked_after` (the same count, which must be 0).

In [ ]:
naive_train, naive_test = ...
leaked_files = ...
splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=0)
train_index, test_index = ...
train_rows, test_rows = photos.iloc[train_index], photos.iloc[test_index]
leaked_after = ...
print(leaked_files, leaked_after)

In [ ]:
check("a random split of files leaks most test files", lambda: leaked_files > 0.8 * len(naive_test), "Count test rows whose source_id also appears in the training rows.")
check("the grouped split leaks nothing", lambda: leaked_after == 0)
check("the grouped split is about 75/25", lambda: 0.2 < len(test_rows) / len(photos) < 0.3)
check("train and test together are the whole table", lambda: len(train_rows) + len(test_rows) == len(photos))
summary()

## Task 2: How much can 300 photos say?

A model got 265 of 300 test photos right, another 255 of 300. Compute each accuracy `acc_a`, `acc_b`, the 95% half-width `half = 1.96 * sqrt(acc * (1 - acc) / 300)` for each (`half_a`, `half_b`),
and set `overlap` to True if the two intervals (accuracy plus or minus half-width) overlap.

In [ ]:
n = 300
acc_a, acc_b = ..., ...
half_a, half_b = ..., ...
overlap = ...
print(acc_a, half_a, acc_b, half_b, overlap)

In [ ]:
check("the accuracies are 0.883 and 0.850", lambda: close([acc_a, acc_b], [0.8833, 0.85], 0.001))
check("each interval is about plus or minus 3.5 points", lambda: close([half_a, half_b], [0.0360, 0.0404], 0.001))
check("the intervals overlap: 300 photos cannot separate these two models", lambda: overlap is True)
summary()